# Tutorial 2
This tutorial covers, how to generate a tight binding file for a heteroatomic system (MoS2) using the Slater-Koster rules

## 1. Generating the basis functions

Numerical atomic orbitals (NAO) serve as basis functions, generated with an atomic DFT calculation. Spherical confinement potentials can be used to tune the basis functions to match properties like a band structure. 

Import modules to handle confinement, atomic DFT, H/S Slater-Koster tables and dipole Slater-Koster tables

In [ ]:
import sys, os
path = os.path.abspath("../..")
if path not in sys.path:
    sys.path.insert(0, path)
for p in sys.path:
    print(p)

from hotcent.confinement import PowerConfinement 
from hotcent.atomic_dft import AtomicDFT 
from hotcent.pos_op.offsite_twocenter_new import Offsite2cTable # for H/S
from hotcent.pos_op.offsite_twocenter_posop import Offsite2cTablePosOp # for r
from hotcent.pos_op.onsite_momentum import onsite_momentum 

/home/juliuswillrich/Dokumente/phd/hotcent_fork_dipole
/home/juliuswillrich/Dokumente/phd/hotcent_fork_dipole/tutorials/tutorial_1
/nix/store/q4sn9c9mfcv6x43a86pkl24gbk32iw7c-python3-3.13.11-env/lib/python313.zip
/nix/store/q4sn9c9mfcv6x43a86pkl24gbk32iw7c-python3-3.13.11-env/lib/python3.13
/nix/store/q4sn9c9mfcv6x43a86pkl24gbk32iw7c-python3-3.13.11-env/lib/python3.13/lib-dynload

/nix/store/q4sn9c9mfcv6x43a86pkl24gbk32iw7c-python3-3.13.11-env/lib/python3.13/site-packages



Set up atomic DFT and run, save eigenvalues of weakly confined atom (for DFT convergence) and run again with stronger confinement:

In [2]:

xc='GGA_X_PBE+GGA_C_PBE'
rcovS = 3.9
rcovMo = 4.3
wf_confS = {'3s': PowerConfinement(r0=rcovS, s=4.6),
           '3p': PowerConfinement(r0=rcovS, s=4.6),
           '3d': PowerConfinement(r0=rcovS, s=4.6),
           }
wf_confMo = {'4d': PowerConfinement(r0=rcovMo, s=11.6),
           '5s': PowerConfinement(r0=rcovMo, s=11.6),
           '5p': PowerConfinement(r0=rcovMo, s=11.6),
           }

atomS = AtomicDFT('S',
                xc = xc,
                perturbative_confinement=False,
                confinement=PowerConfinement(r0=50, s=4),
                configuration='[Ne] 3s2 3p4 3d0',
                valence=['3s', '3p', '3d'], 
                scalarrel=True,
                maxiter=2500,
                timing=False,
                nodegpts=2500,
                mix=0.2,
                txt='-',
                rmax=500,
                )
atomS.run()
print(atomS.enl)
eigenvaluesS = atomS.enl


atomMo = AtomicDFT('Mo',
                xc = xc,
                perturbative_confinement=False,
                configuration='[Kr] 4d4 5s2 5p0',
                valence=['4d', '5s', '5p'],
                confinement=PowerConfinement(r0=50, s=4),
                scalarrel=True,
                maxiter=2500,
                timing=False,
                # nodegpts=150,
                mix=0.2,
                txt='-',
                rmax=100,
                )
atomMo.run()
print(atomMo.enl)
eigenvaluesMo = atomMo.enl

*********************************************************************
Scalar-relativistic all-electron GGA_X_PBE+GGA_C_PBE calculator for S
*********************************************************************
max 2 nodes, 7500 grid points
Applying PowerConfinement(r0=50.000000, s=4.000000, ubound=1.0e+04)
to get the confined electron density
and the confined 3s and 3p and 3d subshell(s)

Start iteration...
iter   0, dn=1.7e+01>1.0e-07, max 11 sp-iter
iter  10, dn=3.9e-01>1.0e-07, max 6 sp-iter
iter  20, dn=7.4e-03>1.0e-07, max 4 sp-iter
iter  30, dn=3.6e-04>1.0e-07, max 3 sp-iter
iter  40, dn=3.8e-05>1.0e-07, max 2 sp-iter
iter  50, dn=4.1e-06>1.0e-07, max 2 sp-iter
iter  60, dn=4.6e-07>1.0e-07, max 1 sp-iter
iter  70, dn=5.2e-08>1.0e-07, max 1 sp-iter

valence subshell eigenvalues:
-----------------------------
  3s:   -0.633837833618
  3p:   -0.257585555567
  3d:   0.033240968433

energy contributions:
----------------------------------------
sum of eigenvalues:     -229.10612824452

In [3]:

print(atomS.enl)
print(atomMo.enl)

{'1s': np.float64(-88.37102145632778), '2s': np.float64(-7.7769037520361), '3s': np.float64(-0.633837833617625), '2p': np.float64(-5.752043323049522), '3p': np.float64(-0.25758555556709223), '3d': np.float64(0.03324096843303342)}
{'1s': np.float64(-728.0787770291039), '2s': np.float64(-102.94354350237869), '3s': np.float64(-17.6124326945093), '4s': np.float64(-2.464476603164679), '5s': np.float64(-0.1791035616091237), '2p': np.float64(-92.09012021225735), '3p': np.float64(-14.047084684869267), '4p': np.float64(-1.510324387437072), '5p': np.float64(-0.052471248271378244), '3d': np.float64(-8.23115087683612), '4d': np.float64(-0.20776147984897222)}


In [4]:

atomS.set_wf_confinement(wf_confinement=wf_confS)
atomS.run()

atomMo.set_wf_confinement(wf_confinement=wf_confMo)
atomMo.run()

Applying PowerConfinement(r0=3.900000, s=4.600000, ubound=1.0e+04)
to get a confined 3s subshell

Start iteration...
iter   0, dn=1.7e+01>1.0e-07, max 11 sp-iter
iter  10, dn=3.0e-01>1.0e-07, max 5 sp-iter
iter  20, dn=9.9e-03>1.0e-07, max 3 sp-iter
iter  30, dn=3.7e-04>1.0e-07, max 3 sp-iter
iter  40, dn=2.0e-05>1.0e-07, max 2 sp-iter
iter  50, dn=1.6e-06>1.0e-07, max 2 sp-iter
iter  60, dn=1.4e-07>1.0e-07, max 1 sp-iter
iter  70, dn=1.3e-08>1.0e-07, max 1 sp-iter

valence subshell eigenvalues:
-----------------------------
  3s:   -0.435738752470
  3p:   -0.032041598355
  3d:   0.528263790308

energy contributions:
----------------------------------------
sum of eigenvalues:     -225.120647556787
Hartree energy:         180.840335354034
vxc correction:         -33.248568810768
exchange-corr. energy:  -25.727440733396
----------------------------------------
total energy:           -398.439854833449

converged in 72 iterations
  16.0001 electrons, should be   16.0000
Applying PowerCon

In [5]:

print(atomS.enl)
print(atomMo.enl)

{'3s': np.float64(-0.435738752470131), '3p': np.float64(-0.03204159835453203), '3d': np.float64(0.5282637903083003), '1s': np.float64(-88.37102145632778), '2s': np.float64(-7.7769037520361), '2p': np.float64(-5.752043323049522)}
{'4d': np.float64(-0.03651353682302553), '5s': np.float64(0.054255777981327434), '5p': np.float64(0.2955575842506935), '1s': np.float64(-728.0787770291039), '2s': np.float64(-102.94354350237869), '3s': np.float64(-17.6124326945093), '4s': np.float64(-2.464476603164679), '2p': np.float64(-92.09012021225735), '3p': np.float64(-14.047084684869267), '4p': np.float64(-1.510324387437072), '3d': np.float64(-8.23115087683612)}


Compute Slater-Koster tables for overlap and Hamiltonian for all possible atom interactions

In [6]:

rmin, dr, N = 0.4, 0.02, 900

off2cMoS = Offsite2cTable(atomS, atomMo, timing=True)
off2cMoS.run(rmin, dr, N, xc=xc, nr=200, ntheta=400, wflimit=1e-9)
off2cMoS.write(format='unique') 

onsite_momentum(atomS, "S_p.skf")
off2cS = Offsite2cTable(atomS, atomS, timing=True)
off2cS.run(rmin, dr, N, xc=xc, nr=200, ntheta=400, wflimit=1e-9)
off2cS.write(format='unique', eigenvalues=eigenvaluesS) 


onsite_momentum(atomMo, "Mo_p.skf")
off2cMo = Offsite2cTable(atomMo, atomMo, timing=True)
off2cMo.run(rmin, dr, N, xc=xc, nr=200, ntheta=400, wflimit=1e-9)
off2cMo.write(format='unique', eigenvalues=eigenvaluesMo) 

Wave function range for S = 8.15222 a0
Wave function range for Mo = 6.58635 a0
R=    0.40, 69430 grid points ...
R=    2.20, 93854 grid points ...
R=    4.00, 103996 grid points ...
R=    5.80, 107348 grid points ...
R=    7.60, 100430 grid points ...
R=    9.40, 34996 grid points ...
R=   11.20, 17118 grid points ...
R=   13.00, 8020 grid points ...
R=   14.80, 2566 grid points ...
Writing to S-Mo.skf
Writing to Mo-S.skf
hermitian up  to 7.146791258794138e-08
Wave function range for S = 8.15222 a0
R=    0.40, 69430 grid points ...
R=    2.20, 93854 grid points ...
R=    4.00, 103996 grid points ...
R=    5.80, 107348 grid points ...
R=    7.60, 100430 grid points ...
R=    9.40, 34996 grid points ...
R=   11.20, 17118 grid points ...
R=   13.00, 8020 grid points ...
R=   14.80, 2566 grid points ...
Writing to S-S.skf
hermitian up  to 1.6505693145596467e-08
Wave function range for Mo = 6.58635 a0
R=    0.40, 71664 grid points ...
R=    2.20, 97620 grid points ...
R=    4.00, 106622 gri

Calculate Slater-Koster tables for position operator elements for all possible atom interactions

In [7]:

off2c_dipoleS = Offsite2cTablePosOp(atomS, atomS, timing=False)
off2c_dipoleS.run(rmin, dr, N, nr=200, ntheta=400, wflimit=1e-9)
off2c_dipoleS.write_dipole(format='unique')

off2c_dipoleMo = Offsite2cTablePosOp(atomMo, atomMo, timing=False)
off2c_dipoleMo.run(rmin, dr, N, nr=200, ntheta=400, wflimit=1e-9)
off2c_dipoleMo.write_dipole(format='unique')

off2c_dipoleMoS = Offsite2cTablePosOp(atomMo, atomS, timing=False)
off2c_dipoleMoS.run(rmin, dr, N, nr=200, ntheta=400, wflimit=1e-9)
off2c_dipoleMoS.write_dipole(format='unique')

Wave function range for S = 8.15222 a0
R=    0.40, 69430 grid points ...
R=    2.20, 93854 grid points ...
R=    4.00, 103996 grid points ...
R=    5.80, 107348 grid points ...
R=    7.60, 100430 grid points ...
R=    9.40, 34996 grid points ...
R=   11.20, 17118 grid points ...
R=   13.00, 8020 grid points ...
R=   14.80, 2566 grid points ...
Wave function range for Mo = 6.58635 a0
R=    0.40, 71664 grid points ...
R=    2.20, 97620 grid points ...
R=    4.00, 106622 grid points ...
R=    5.80, 103918 grid points ...
R=    7.60, 34904 grid points ...
R=    9.40, 14536 grid points ...
R=   11.20, 5038 grid points ...
R=   13.00, 278 grid points ...
Wave function range for Mo = 6.58635 a0
Wave function range for S = 8.15222 a0
R=    0.40, 69430 grid points ...
R=    2.20, 93854 grid points ...
R=    4.00, 103996 grid points ...
R=    5.80, 107348 grid points ...
R=    7.60, 100430 grid points ...
R=    9.40, 34996 grid points ...
R=   11.20, 17118 grid points ...
R=   13.00, 8020 grid p

## Important: Set energy eigenvalue of sulfur d orbital to 0.134677 manually
Move parameter files in separate directories for H/S and for position operator respectively
Additional directory for onsite momentum

In [8]:
!mkdir skfiles
!mkdir skfiles_posop
!mkdir onsite_momentum
!mv S-S_dipole.skf skfiles_posop/S-S.skf
!mv Mo-Mo_dipole.skf skfiles_posop/Mo-Mo.skf
!mv S-S.skf skfiles/
!mv S_p.skf onsite_momentum
!mv Mo_p.skf onsite_momentum

# 3. Generating the tight binding parameters for a specific system

Based on the Slater-Koster parameter files, the matrix elements for overlap, Hamiltonian and dipole elements can be calculated for arbitrary molecular geometries. All interactions within the cutoff of the Slater-Koster file are considered. The output can be generated in a non-orthogonal tight-binding format similar to the one used by Wannier90.

In order to do that, create a directory for the SK files and another one for the dipole SK files. And remove the _dipole suffix. Additionally a geometry file for the system of interest that can be read by `ase` is required. 

In [1]:
from ase.build import mx2
from hotcent.pos_op.mat_elem_evaluation import SlaterKosterIntegrator
MoS2 = mx2(vacuum=20)
MoS2.pbc = (True, True, True)
max_l = {'C':1, 'H':0, 'S':2, 'Mo':2}
seedname_mos2 = SlaterKosterIntegrator(
    MoS2,
    skpath="skfiles",
    maxl_dict=max_l,
    skpath_posop="skfiles_posop",
    format="unique",
    path_p_onsite="onsite_momentum"
)
seedname_mos2.write_seedname()

ModuleNotFoundError: No module named 'hotcent'

For momentum

In [ ]:
seedname_mos2.write_seedname_momentum()